===TITLE===

As word, stems, or lemma? Embeddings?

MAIN
* title embeddings (word2vec, Bert, glove, fastext)
* Number of characters in the title.
* Number of words in the title.
* Title embeddings?
* Title TFIDF, count, binary
* Sentiment score

CAPITALIZATION
* Number of capital letters
* Percent capital letters
* Number all caps words
* Percent all caps words
* Specific capitalized words
* Specific capitalized parts of speech
* Consecutive capital letters [and spaces or punctuation]

PUNCTUATION
* Number of exclamation points
* Percent exclamation points
* Number question marks
* Percent question marks
* Number special
* Percent special
* Number punctuation
* Percent punctuation
* Number of digits
* Percent digits
* Number of numbers
* Percent numbers
* Number Different types of punctuation
	
COMPLEXITY
* readability (multiple scores)
* Syllable count
* Syllables per word
* Most syllables in a word
* Max vs average syllables
* Letters per word
* Number of parts of speech
* Number of each part
* Number of each as percentage of total
* Ratios comparing each part to one another (lexical density)
* Parts of speech order (specific and broad)
* Type-token ratio / lexical diversity

KEYWORDS
	•	“Specificity score”??? (Tfidf?)
	•	Clickbait measures???
	•	Richness (num or unique keywords vs title length)
	•	Num unique keywords
	•	Percent unique keywords
	•	Key vs nonkey pattern
	•	Keyword length
	•	First keyword position [as percentile]
	•	Keyword by lexical type 
	•	Keywords in (first/last) N words
N GRAMS
	•	binary n gram
	•	Count n gram
	•	Tfidf n gram
SEMANTIC
	•	named entity recognition counts
	•	Named entity recognition types counts (people, places, groups, etc)
	•	Topic modeling (for example, LDA) to assign titles or n grams to topics or clusters
	•	Multiple models?
	•	Somehow tune topic models to find best granularity?
	•	Similarity scores with high performing titles or trending topics
OTHER
	•	Keyword trends over life of video
	•	Keyword trends at video release 
	•	Somehow quantify rhythm or cadence?
	•	Features from models pre trained to identify certain things?
	•	Features representing historical performance of similar videos
	•	Sentiment (pos-neg score)
	•	Sentiment as pos/neutral/neg
	•	Sentiment as binaries (pos, neutral, negative)
	•	Emotion via detection?
	•	Mutual information?

In [ ]:
from sklearn.model_selection import train_test_split

from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor

import pandas as pd
import sqlalchemy

from nltk import pos_tag
from nltk.tokenize import word_tokenize
from collections import Counter

# Create a connection to the database
engine = sqlalchemy.create_engine('sqlite:///../data/data.db')

video_df = pd.read_sql('SELECT * FROM search_results', engine)
title_df = video_df.copy()
title_df = title_df[['id', 'snippet.title', 'statistics.viewCount']]
title_df['statistics.viewCount'] = title_df['statistics.viewCount'].dropna().astype(int)
title_df.sort_values('statistics.viewCount', ascending=False)
title_df.columns = ['id', 'title', 'views']
title_df = title_df.set_index('id')
title_df.head()

In [ ]:
capital_df = title_df.copy()
capital_df['num_capital_letters'] = capital_df['title'].apply(lambda x: sum(1 for c in x if c.isupper()))
capital_df['num_all_caps_words'] = capital_df['title'].apply(lambda x: sum(1 for c in x.split() if c.isupper()))
capital_df['num_capitalized_words'] = capital_df['title'].apply(lambda x: sum(1 for c in x.split() if c.istitle()))
capital_df['pct_capital_letters'] = capital_df['num_capital_letters'] / capital_df['title'].apply(len)
capital_df['pct_all_caps_words'] = capital_df['num_all_caps_words'] / capital_df['title'].apply(lambda x: len(x.split()))
capital_df['pct_capitalized_words'] = capital_df['num_capitalized_words'] / capital_df['title'].apply(lambda x: len(x.split()))
capital_df = capital_df.drop(columns=['title']).dropna()

capital_X = capital_df.drop(columns=['views'])
capital_y = capital_df['views']

capital_X_train, capital_X_test, capital_y_train, capital_y_test = train_test_split(capital_X, capital_y, test_size=0.2, random_state=42)

lasso = Lasso()
lasso.fit(capital_X_train, capital_y_train)
print(f'Lasso R^2: {lasso.score(capital_X_test, capital_y_test)}')

rf = RandomForestRegressor()
rf.fit(capital_X_train, capital_y_train)
print(f'Random Forest R^2: {rf.score(capital_X_test, capital_y_test)}')

In [ ]:
punctuation_df = title_df.copy()
punctuation_df['num_exclamation_points'] = punctuation_df['title'].apply(lambda x: x.count('!'))
punctuation_df['num_question_marks'] = punctuation_df['title'].apply(lambda x: x.count('?'))
punctuation_df['num_special'] = punctuation_df['title'].apply(lambda x: sum(1 for c in x if not c.isalnum()))
punctuation_df['num_punctuation'] = punctuation_df['title'].apply(lambda x: sum(1 for c in x if c in ['!', '?', '.', ',']))
punctuation_df['num_digits'] = punctuation_df['title'].apply(lambda x: sum(1 for c in x if c.isdigit()))
punctuation_df['pct_exclamation_points'] = punctuation_df['num_exclamation_points'] / punctuation_df['title'].apply(len)
punctuation_df['pct_question_marks'] = punctuation_df['num_question_marks'] / punctuation_df['title'].apply(len)
punctuation_df['pct_special'] = punctuation_df['num_special'] / punctuation_df['title'].apply(len)
punctuation_df['pct_punctuation'] = punctuation_df['num_punctuation'] / punctuation_df['title'].apply(len)
punctuation_df['pct_digits'] = punctuation_df['num_digits'] / punctuation_df['title'].apply(len)
punctuation_df = punctuation_df.drop(columns=['title']).dropna()

punctuation_X = punctuation_df.drop(columns=['views'])
punctuation_y = punctuation_df['views']

punctuation_X_train, punctuation_X_test, punctuation_y_train, punctuation_y_test = train_test_split(punctuation_X, punctuation_y, test_size=0.2, random_state=42)

lasso = Lasso()
lasso.fit(punctuation_X_train, punctuation_y_train)
print(f'Lasso R^2: {lasso.score(punctuation_X_test, punctuation_y_test)}')

rf = RandomForestRegressor()
rf.fit(punctuation_X_train, punctuation_y_train)
print(f'Random Forest R^2: {rf.score(punctuation_X_test, punctuation_y_test)}')

In [ ]:
pd.set_option('display.max_colwidth', 64)
pd.set_option('display.width', 256)

descriptive_df = title_df.copy()
descriptive_df['num_words'] = descriptive_df['title'].apply(lambda x: len(x.split()))
descriptive_df['num_characters'] = descriptive_df['title'].apply(len)
descriptive_df['num_vowels'] = descriptive_df['title'].apply(lambda x: sum(1 for c in x if c.lower() in 'aeiouy'))
descriptive_df['num_consonants'] = descriptive_df['title'].apply(lambda x: sum(1 for c in x if c.isalpha() and c.lower() not in 'aeiouy'))
descriptive_df['avg_word_length'] = descriptive_df['num_characters'] / descriptive_df['num_words']
descriptive_df['vowels_per_word'] = descriptive_df['num_vowels'] / descriptive_df['num_words']
descriptive_df['vowels per consonant'] = descriptive_df['num_vowels'] / descriptive_df['num_consonants']
descriptive_df['min_word_length'] = descriptive_df['title'].apply(lambda x: min(len(word) for word in x.split()))
descriptive_df['max_word_length'] = descriptive_df['title'].apply(lambda x: max(len(word) for word in x.split()))
descriptive_df['max_word_length / min_word_length'] = descriptive_df['max_word_length'] / descriptive_df['min_word_length']
descriptive_df['avg_word_length / min_word_length'] = descriptive_df['avg_word_length'] / descriptive_df['min_word_length']
descriptive_df['max_word_length / avg_word_length'] = descriptive_df['max_word_length'] / descriptive_df['avg_word_length']
descriptive_df['min_num_vowels_per_word'] = descriptive_df['title'].apply(lambda x: min(sum(1 for c in word if c in 'aeiouy') for word in x.split()))
descriptive_df['max_num_vowels_per_word'] = descriptive_df['title'].apply(lambda x: max(sum(1 for c in word if c in 'aeiouy') for word in x.split()))
descriptive_df['max_num_vowels_per_word / min_num_vowels_per_word'] = descriptive_df['max_num_vowels_per_word'] / descriptive_df['min_num_vowels_per_word']
descriptive_df['vowels_per_word / min_num_vowels_per_word'] = descriptive_df['vowels_per_word'] / descriptive_df['min_num_vowels_per_word']
descriptive_df['max_num_vowels_per_word / vowels_per_word'] = descriptive_df['max_num_vowels_per_word'] / descriptive_df['vowels_per_word']
descriptive_df = descriptive_df.drop(columns=['title']).dropna()

# drop inf
descriptive_df = descriptive_df.replace([float('inf'), float('-inf')], float('nan')).dropna()

descriptive_X = descriptive_df.drop(columns=['views'])
descriptive_y = descriptive_df['views']

descriptive_X_train, descriptive_X_test, descriptive_y_train, descriptive_y_test = train_test_split(descriptive_X, descriptive_y, test_size=0.2, random_state=42)

lasso = Lasso(max_iter=100000)
lasso.fit(descriptive_X_train, descriptive_y_train)
print(f'Lasso R^2: {lasso.score(descriptive_X_test, descriptive_y_test)}')

rf = RandomForestRegressor()
rf.fit(descriptive_X_train, descriptive_y_train)
print(f'Random Forest R^2: {rf.score(descriptive_X_test, descriptive_y_test)}')

In [ ]:
reading_df = title_df.copy()
reading_df['flesch_reading_ease'] = reading_df['title'].apply(lambda x: 206.835 - 1.015 * len(x.split()) - 84.6 * sum(1 for c in x if c.lower() in 'aeiouy') / len(x.split()))
reading_df['flesch_kincaid_grade'] = reading_df['title'].apply(lambda x: 0.39 * len(x.split()) + 11.8 * sum(1 for c in x if c.lower() in 'aeiouy') / len(x.split()) - 15.59)
reading_df['gunning_fog_index'] = reading_df['title'].apply(lambda x: 0.4 * (len(x.split()) + 100 * sum(1 for c in x if c in ['.', '!', '?'])) / len(x.split()))
reading_df['smog_index'] = reading_df['title'].apply(lambda x: 1.043 * (30 * sum(1 for c in x if c in ['.', '!', '?']) / len(x.split())) ** 0.5 + 3.1291)
reading_df['coleman_liau_index'] = reading_df['title'].apply(lambda x: 5.88 * sum(1 for c in x if c.isalnum()) / len(x) - 0.296 * 100 / len(x.split()) - 15.8)
reading_df['automated_readability_index'] = reading_df['title'].apply(lambda x: 4.71 * len(x) / len(x.split()) + 0.5 * len(x.split()) - 21.43)
reading_df['linsear_write_formula'] = reading_df['title'].apply(lambda x: (sum(1 for c in x if c.lower() in 'aeiouy') + sum(1 for c in x if c in ['.', '!', '?'])) / len(x.split()) / 2)
reading_df['dale_chall_readability_score'] = reading_df['title'].apply(lambda x: 0.1579 * (100 * sum(1 for c in x if c.isalnum()) / len(x)) + 0.0496 * len(x.split()))
reading_df = reading_df.drop(columns=['title']).dropna()

reading_X = reading_df.drop(columns=['views'])
reading_y = reading_df['views']

reading_X_train, reading_X_test, reading_y_train, reading_y_test = train_test_split(reading_X, reading_y, test_size=0.2, random_state=42)

lasso = Lasso()
lasso.fit(reading_X_train, reading_y_train)
print(f'Lasso R^2: {lasso.score(reading_X_test, reading_y_test)}')

rf = RandomForestRegressor(max_depth=100)
rf.fit(reading_X_train, reading_y_train)
print(f'Random Forest R^2: {rf.score(reading_X_test, reading_y_test)}')

In [ ]:
from nltk.tag import pos_tag, map_tag

pos_tags = title_df['title'].apply(lambda x: Counter(tag for word, tag in pos_tag(word_tokenize(x))))

specific_pos_tags_count_df = pd.DataFrame(list(pos_tags), index=title_df.index)
specific_pos_tags_count_df = specific_pos_tags_count_df.fillna(0)

for col in specific_pos_tags_count_df.columns:
    if not col.isalnum():
        specific_pos_tags_count_df.drop(columns=col, inplace=True)

generic_pos_tags_count_df = specific_pos_tags_count_df.copy()
generic_pos_tags_count_df['noun'] = generic_pos_tags_count_df[['NN', 'NNS', 'NNP', 'NNPS']].sum(axis=1)
generic_pos_tags_count_df['verb'] = generic_pos_tags_count_df[['VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ']].sum(axis=1)
generic_pos_tags_count_df['adjective'] = generic_pos_tags_count_df[['JJ', 'JJR', 'JJS']].sum(axis=1)
generic_pos_tags_count_df['adverb'] = generic_pos_tags_count_df[['RB', 'RBR', 'RBS']].sum(axis=1)
generic_pos_tags_count_df['pronoun'] = generic_pos_tags_count_df[['PRP', 'WP']].sum(axis=1)
generic_pos_tags_count_df['preposition'] = generic_pos_tags_count_df[['IN', 'TO']].sum(axis=1)
generic_pos_tags_count_df['conjunction'] = generic_pos_tags_count_df[['CC']].sum(axis=1)
generic_pos_tags_count_df['interjection'] = generic_pos_tags_count_df[['UH']].sum(axis=1)
generic_pos_tags_count_df['determiner'] = generic_pos_tags_count_df[['DT', 'PDT', 'WDT']].sum(axis=1)
generic_pos_tags_count_df = generic_pos_tags_count_df[['noun', 'verb', 'adjective', 'adverb', 'pronoun', 'preposition', 'conjunction', 'interjection', 'determiner']]

specific_pos_tags_pct_df = specific_pos_tags_count_df.div(specific_pos_tags_count_df.sum(axis=1), axis=0)
generic_pos_tags_pct_df = generic_pos_tags_count_df.div(generic_pos_tags_count_df.sum(axis=1), axis=0)

specific_pos_tags_count_df.columns = ['specific_count.' + col for col in specific_pos_tags_count_df.columns]
specific_pos_tags_pct_df.columns = ['specific_pct.' + col for col in specific_pos_tags_pct_df.columns]
generic_pos_tags_count_df.columns = ['generic_count.' + col for col in generic_pos_tags_count_df.columns]
generic_pos_tags_pct_df.columns = ['generic_pct.' + col for col in generic_pos_tags_pct_df.columns]

specific_tags_df = pd.concat([specific_pos_tags_count_df, specific_pos_tags_pct_df], axis=1)
generic_tags_df = pd.concat([generic_pos_tags_count_df, generic_pos_tags_pct_df], axis=1)

count_tags_df = pd.concat([specific_pos_tags_count_df, generic_pos_tags_count_df], axis=1)
pct_tags_df = pd.concat([specific_pos_tags_pct_df, generic_pos_tags_pct_df], axis=1)

all_pos_tags_df = pd.concat([specific_pos_tags_count_df, specific_pos_tags_pct_df, generic_pos_tags_count_df, generic_pos_tags_pct_df], axis=1).fillna(0)

In [ ]:
import nltk
nltk.download('vader_lexicon')

# score sentiment
from nltk.sentiment.vader import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

sentiment_df = title_df.copy()
sentiment_df['sentiment'] = sentiment_df['title'].apply(lambda x: sia.polarity_scores(x)['compound'])

# get sentiment of each word
def get_word_sentiment(title):
    return [sia.polarity_scores(word)['compound'] for word in title.split()]

word_sentiment = title_df['title'].apply(get_word_sentiment)
sentiment_df['avg_word_sentiment'] = word_sentiment.apply(lambda x: sum(x) / len(x))
sentiment_df['max_word_sentiment'] = word_sentiment.apply(max)
sentiment_df['min_word_sentiment'] = word_sentiment.apply(min)
sentiment_df = sentiment_df.drop(columns=['title'])

# pad with zeros
max_len = max(len(sent) for sent in word_sentiment)
word_sentiment = word_sentiment.apply(lambda x: x + [0] * (max_len - len(x)))
word_sentiment_df = pd.DataFrame(word_sentiment.tolist(), index=word_sentiment.index)
word_sentiment_df.columns = ['word.' + str(i) for i in word_sentiment_df.columns] 

DO NOT TOUCH ABOVE

In [ ]:
# add to path
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

dfs = [title_df, capital_df, punctuation_df, descriptive_df, reading_df, all_pos_tags_df, sentiment_df, word_sentiment_df]


X_df = pd.concat(dfs, axis=1).fillna(0)
y_df = title_df['views'].dropna()
X_df = X_df.loc[y_df.index]

# tokenize the title
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def tokenize(title):
    return [word.lower() for word in word_tokenize(title) if word.isalnum() and word not in stop_words]

tokenized_titles = title_df['title'].apply(tokenize)

# lemmatize the title
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize(title):
    return [lemmatizer.lemmatize(word) for word in title]

lemmatized_titles = tokenized_titles.apply(lemmatize)

# convert words like 𝙄𝙈𝙋𝙊𝙎𝙎𝙄𝘽𝙇𝙀 to IMPOSSIBLE
from unidecode import unidecode

def remove_accented_chars(title):
    return [unidecode(word) for word in title]

cleaned_titles = lemmatized_titles.apply(remove_accented_chars)

title_strings = cleaned_titles.apply(' '.join)

X_df['title'] = title_strings

X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size=0.2, random_state=42)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

max_features = int(np.sqrt(len(X_train)))
min_features = int(np.log(len(X_train)))
tfidf = TfidfVectorizer(stop_words='english', min_df=min_features, max_df=max_features)
tfidf.fit(X_train['title'])

X_train_tfidf = tfidf.transform(X_train['title'])
X_test_tfidf = tfidf.transform(X_test['title'])

X_train_tfidf_df = pd.DataFrame(X_train_tfidf.toarray(), index=X_train.index, columns=tfidf.get_feature_names_out())
X_test_tfidf_df = pd.DataFrame(X_test_tfidf.toarray(), index=X_test.index, columns=tfidf.get_feature_names_out())

X_train = pd.concat([X_train.drop(columns=['title'], errors='ignore'), X_train_tfidf_df], axis=1)
X_test = pd.concat([X_test.drop(columns=['title'], errors='ignore'), X_test_tfidf_df], axis=1)

X_train.drop(columns=['views'], inplace=True, errors='ignore')
X_test.drop(columns=['views'], inplace=True, errors='ignore')

# drop cols if all zeros
X_train = X_train.loc[:, (X_train != 0).any(axis=0)]
X_test = X_test[X_train.columns]

X_train.shape, X_test.shape

In [10]:
lr = Lasso(alpha=.001, max_iter=1000)
lr.fit(X_train, y_train)
print(f'Lasso R^2: {lr.score(X_test, y_test)}')

# rf = RandomForestRegressor()
# rf.fit(X_train, y_train)
# print(f'Random Forest R^2: {rf.score(X_test, y_test)}')

# coefs = pd.Series(lr.coef_, index=X_train.columns)
# imps = pd.Series(rf.feature_importances_, index=X_train.columns)
# features = pd.DataFrame({'coefs': coefs, 'imps': imps})
# features = features[features['coefs'] * features['imps'] != 0]
# features.sort_values('imps', ascending=False)

In [ ]:
# add to path
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

from Selector import CombineCorrelatedFeaturesSelector, MutualInforegressionSelector, VIFSelector, CooksDistanceInfluenceSelector

# combine correlated features
ccfs = CombineCorrelatedFeaturesSelector(threshold=1)
ccfs.fit(X_train, y_train)
X_train_ccfs, y_train = ccfs.transform(X_train, y_train)
X_test_ccfs, y_train = ccfs.transform(X_test, y_train)

# Mutual Information Regression
mir = MutualInforegressionSelector()
mir.fit(X_train, y_train)
X_train_mir, y_train = mir.transform(X_train, y_train)
X_test_mir, y_train = mir.transform(X_test, y_train)

X_train_mir.shape, X_test_mir.shape

In [ ]:
ccfs_aggro = CombineCorrelatedFeaturesSelector(threshold=0.8)
ccfs_aggro.fit(X_train_mir, y_train)
X_train_ccfs_aggro, y_train = ccfs_aggro.transform(X_train_mir, y_train)
X_test_ccfs_aggro, y_train = ccfs_aggro.transform(X_test_mir, y_train)

vif = VIFSelector(threshold=5)
vif.fit(X_train_ccfs_aggro, y_train)
X_train_vif, y_train = vif.transform(X_train_ccfs_aggro, y_train)
X_test_vif, y_train = vif.transform(X_test_ccfs_aggro, y_train)

X_train_vif.shape, X_test_vif.shape

In [ ]:
lr = Lasso()
lr.fit(X_train_vif, y_train)
print(f'Lasso R^2: {lr.score(X_test_vif, y_test)}')

In [ ]:
rf = RandomForestRegressor()
rf.fit(X_train_vif, y_train)
print(f'Random Forest R^2: {rf.score(X_test_vif, y_test)}')

In [ ]:
# lasso = Lasso()
# lasso.fit(X_train_tfidf_df, y_train)
# print(f'Lasso R^2: {lasso.score(X_test_tfidf_df, y_test)}')

# rf = RandomForestRegressor()
# rf.fit(X_train_tfidf_df, y_train)
# print(f'Random Forest R^2: {rf.score(X_test_tfidf_df, y_test)}')

In [ ]:
# dont use scientific notation
pd.set_option('display.float_format', lambda x: '%.3f' % x)

coefs = pd.Series(lr.coef_, index=X_train_vif.columns)
imps = pd.Series(rf.feature_importances_, index=X_train_vif.columns)
features = pd.concat([coefs, imps], axis=1)
features.columns = ['coefs', 'imps']
feautures = features[features['coefs'] * features['imps'] != 0]
features.sort_values('imps', ascending=False)

In [ ]:
new_X_train, new_X_test = X_train[features.index], X_test[features.index]

lasso = Lasso()
lasso.fit(new_X_train, y_train)
print(f'Lasso R^2: {lasso.score(new_X_test, y_test)}')

rf = RandomForestRegressor()
rf.fit(new_X_train, y_train)
print(f'Random Forest R^2: {rf.score(new_X_test, y_test)}')

In [ ]:
# drop_cols = ['num_capital_letters', 'num_all_caps_words', 'num_capitalized_words']

# # drop columns if not alphanumeric
# for col in X_df.columns:
#     if not col.isalnum() and 'word' not in col:
#         drop_cols.append(col)

# X_df = X_df.drop(columns=drop_cols, errors='ignore')

# lasso = Lasso()
# lasso.fit(X_train_poly, y_train)
# print(f'Lasso R^2: {lasso.score(X_test_poly, y_test)}')

# rf = RandomForestRegressor()
# rf.fit(X_train_poly, y_train)
# print(f'Random Forest R^2: {rf.score(X_test_poly, y_test)}')

In [ ]:
# from Selector import CombineCorrelatedFeaturesSelector, MutualInforegressionSelector, VIFSelector, CooksDistanceInfluenceSelector

# print('ccfs')
# ccfs = CombineCorrelatedFeaturesSelector(threshold=0.9)
# ccfs.fit(X_train, y_train)

# X_train, y_train = ccfs.transform(X_train, y_train)
# X_test, y_test = ccfs.transform(X_test, y_test)
# print(X_train.shape[1])

# print('mir')
# mir = MutualInforegressionSelector()
# mir.fit(X_train, y_train)

# X_train, y_train = mir.transform(X_train, y_train)
# X_test, y_test = mir.transform(X_test, y_test)
# print(X_train.shape[1])

# print('vif')
# vif = VIFSelector(threshold=5)
# vif.fit(X_train, y_train)

# X_train, y_train = vif.transform(X_train, y_train)
# X_test, y_test = vif.transform(X_test, y_test)
# print(X_train.shape[1])

# print('cds')
# cds = CooksDistanceInfluenceSelector()
# cds.fit(X_train, y_train)

# X_train, y_train = cds.transform(X_train, y_train)
# X_test, y_test = cds.transform(X_test, y_test)

In [ ]:
# lasso = Lasso()
# lasso.fit(X_train, y_train)
# print(f'Lasso R^2: {lasso.score(X_test, y_test)}')

# rf = RandomForestRegressor()
# rf.fit(X_train, y_train)
# print(f'Random Forest R^2: {rf.score(X_test, y_test)}')

In [ ]:
# # dont use scientific notation
# pd.set_option('display.float_format', lambda x: '%.5f' % x)

# lasso_coefs = pd.Series(lasso.coef_, index=X_df.columns)
# rf_importances = pd.Series(rf.feature_importances_, index=X_df.columns)
# features = pd.concat([lasso_coefs, rf_importances], axis=1)
# features.columns = ['lasso', 'rf']
# features = features[features['lasso'] * features['rf'] != 0]
# features.sort_values('rf', ascending=False)